# NYC Social Services — Knowledge Graph Builder (Colab H100/A100)

**Run this notebook on Colab Pro with H100 or A100 runtime.**

Steps:
1. Install RAPIDS (cuDF + cuGraph)
2. Mount Google Drive (data lives there)
3. Build the full 7K-node knowledge graph with cuGraph
4. Save `graph.pkl` back to Drive

After this notebook completes, copy `graph.pkl` to USB for the hackathon venue.

## Cell 1 — Install RAPIDS
**Run once. Restart runtime after this cell completes.**

In [ ]:
# Install RAPIDS cuDF + cuGraph (26.2 — compatible with CUDA 12.x on Colab)
# After this cell finishes, click Runtime → Restart session, then skip to Cell 2.

import subprocess, sys

result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--extra-index-url=https://pypi.nvidia.com",
        "cudf-cu12==25.4.*",
        "cugraph-cu12==25.4.*",
        "cupy-cuda12x",
    ],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if result.stdout else "")
print(result.stderr[-2000:] if result.stderr else "")
print("\nDone. NOW RESTART THE RUNTIME (Runtime → Restart session).")

## Cell 2 — Mount Drive + verify GPU

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)

# Verify cuDF + cuGraph
import cudf, cugraph
print(f"cuDF  version: {cudf.__version__}")
print(f"cuGraph version: {cugraph.__version__}")

## Cell 3 — Set paths
**Edit `DRIVE_DATA_DIR` to point to where you uploaded the `data/` and `stage/` folders.**

In [ ]:
from pathlib import Path

# ⬇️ EDIT THIS — path inside your Google Drive
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/nyc_hack_data")

DATA_DIR  = DRIVE_DATA_DIR / "data"
STAGE_DIR = DRIVE_DATA_DIR / "stage"

# Verify files exist
mart_path = DATA_DIR / "resource_mart.parquet"
assert mart_path.exists(), f"Missing: {mart_path}\nUpload data/ and stage/ folders to Drive first."
print(f"✓ resource_mart.parquet found")

import pandas as pd
mart = pd.read_parquet(mart_path)
print(f"✓ Mart loaded: {len(mart)} rows, {len(mart.columns)} columns")
print(mart['resource_type'].value_counts().to_string())

## Cell 4 — Build the full knowledge graph

In [ ]:
import cudf
import cugraph
import cupy as cp
import numpy as np
import pandas as pd
import pickle
import time
from scipy.spatial import KDTree
from tqdm.notebook import tqdm

LAT_M = 111_320
LON_M = 85_390
RESOURCE_OFFSET = 0
TRANSIT_OFFSET  = 100_000
TRACT_OFFSET    = 200_000

# ── Load data ─────────────────────────────────────────────────────────────
print("Loading data...")
mart = pd.read_parquet(DATA_DIR / "resource_mart.parquet")
mart = mart.dropna(subset=["latitude", "longitude"])

transit_mask = mart["resource_type"] == "transit_station"
transit   = mart[transit_mask].copy().reset_index(drop=True)
resources = mart[~transit_mask].copy().reset_index(drop=True)

resources["node_id"] = RESOURCE_OFFSET + resources.index
transit["node_id"]   = TRANSIT_OFFSET  + transit.index
print(f"Resources: {len(resources)}  Transit: {len(transit)}")

# ── Build tract nodes (1km grid cells) ────────────────────────────────────
res = 0.009
resources["_glat"] = (resources["latitude"]  / res).round() * res
resources["_glon"] = (resources["longitude"] / res).round() * res
tracts = (resources.groupby(["_glat", "_glon"])
          .size().reset_index(name="resource_count"))
tracts = tracts.rename(columns={"_glat": "latitude", "_glon": "longitude"})
tracts["tract_id"] = "synthetic_" + tracts.index.astype(str)
tracts["node_id"]  = TRACT_OFFSET + tracts.index
resources = resources.drop(columns=["_glat", "_glon"])
print(f"Tract nodes: {len(tracts)}")

# ── Build edges with GPU-accelerated KD-tree ───────────────────────────────
print("\nBuilding edges...")
r_coords = np.column_stack([
    resources["latitude"].values  * LAT_M,
    resources["longitude"].values * LON_M,
])
t_coords = np.column_stack([
    transit["latitude"].values  * LAT_M,
    transit["longitude"].values * LON_M,
])
tr_coords = np.column_stack([
    tracts["latitude"].values  * LAT_M,
    tracts["longitude"].values * LON_M,
])

r_tree  = KDTree(r_coords)
t_tree  = KDTree(t_coords)
tr_tree = KDTree(tr_coords)

edge_src, edge_dst, edge_weight, edge_type = [], [], [], []

def add(srcs, dsts, wts, etype):
    edge_src.extend(srcs); edge_dst.extend(dsts)
    edge_weight.extend(wts); edge_type.extend([etype]*len(srcs))

# NEAR edges (k=5, max 2km)
print("  NEAR edges...")
k = min(6, len(resources))
dists, idxs = r_tree.query(r_coords, k=k, workers=-1)
ns, nd, nw = [], [], []
for i, (row_d, row_i) in enumerate(zip(dists, idxs)):
    ni = int(resources.iloc[i]["node_id"])
    for d, j in zip(row_d[1:], row_i[1:]):
        if d <= 2000:
            nj = int(resources.iloc[j]["node_id"])
            ns += [ni, nj]; nd += [nj, ni]; nw += [round(d,1), round(d,1)]
add(ns, nd, nw, "NEAR")
print(f"    {len(ns)} NEAR edges")

# WALK_TO_TRANSIT
print("  WALK_TO_TRANSIT edges...")
t_dists, t_idxs = t_tree.query(r_coords, k=1, workers=-1)
ws, wd, ww = [], [], []
for i, (d, j) in enumerate(zip(t_dists, t_idxs)):
    nr = int(resources.iloc[i]["node_id"])
    nt = int(transit.iloc[j]["node_id"])
    wm = round(d / 80, 2)
    ws += [nr, nt]; wd += [nt, nr]; ww += [wm, wm]
add(ws, wd, ww, "WALK_TO_TRANSIT")
print(f"    {len(ws)} WALK_TO_TRANSIT edges")

# TRANSIT_LINK
print("  TRANSIT_LINK edges...")
if "subway_lines" in transit.columns:
    line_map = {}
    for _, row in transit.iterrows():
        for line in str(row.get("subway_lines", "")).split():
            line_map.setdefault(line, []).append(int(row["node_id"]))
    tls, tld, tlw = [], [], []
    for line, stations in line_map.items():
        for i in range(len(stations) - 1):
            tls += [stations[i], stations[i+1]]
            tld += [stations[i+1], stations[i]]
            tlw += [3.0, 3.0]
    add(tls, tld, tlw, "TRANSIT_LINK")
    print(f"    {len(tls)} TRANSIT_LINK edges")

# IN_TRACT
print("  IN_TRACT edges...")
_, tr_idxs = tr_tree.query(r_coords, k=1, workers=-1)
its, itd, itw = [], [], []
for i, j in enumerate(tr_idxs):
    its.append(int(resources.iloc[i]["node_id"]))
    itd.append(int(tracts.iloc[j]["node_id"]))
    itw.append(1.0)
add(its, itd, itw, "IN_TRACT")
print(f"    {len(its)} IN_TRACT edges")

# SERVED_BY — GPU-accelerated: vectorized over all tracts at once
print("  SERVED_BY edges (GPU vectorized)...")
t0_sb = time.time()

r_lat_m = cp.array(r_coords[:, 0], dtype=cp.float32)
r_lon_m = cp.array(r_coords[:, 1], dtype=cp.float32)
r_walk  = cp.array(t_dists / 80, dtype=cp.float32)  # walk time resource→transit
r_nids  = cp.array(resources["node_id"].values, dtype=cp.int32)

sbs, sbd, sbw = [], [], []
BATCH = 50  # process 50 tracts at a time

for batch_start in tqdm(range(0, len(tracts), BATCH), desc="SERVED_BY batches"):
    batch = tracts.iloc[batch_start:batch_start+BATCH]
    tr_lat_m = cp.array(batch["latitude"].values  * LAT_M, dtype=cp.float32)
    tr_lon_m = cp.array(batch["longitude"].values * LON_M, dtype=cp.float32)
    tr_nids  = cp.array(batch["node_id"].values, dtype=cp.int32)

    # Shape: (n_tracts_in_batch, n_resources)
    # Find nearest transit station to each tract centroid
    tr_coords_np = np.column_stack([batch["latitude"].values * LAT_M,
                                    batch["longitude"].values * LON_M])
    tr_t_dists, _ = t_tree.query(tr_coords_np, k=1)
    tr_walk = cp.array(tr_t_dists / 80, dtype=cp.float32)  # (n_batch,)

    # Straight-line dist from each tract to each resource
    dlat = tr_lat_m[:, None] - r_lat_m[None, :]  # (n_batch, n_resources)
    dlon = tr_lon_m[:, None] - r_lon_m[None, :]
    dist_m = cp.sqrt(dlat**2 + dlon**2)

    subway_time = dist_m / 500.0
    total_time  = tr_walk[:, None] + subway_time + r_walk[None, :]  # (n_batch, n_resources)

    # Only keep resource pairs within 60 min
    mask = total_time <= 60.0

    # Extract edges
    tract_idxs, res_idxs = cp.where(mask)
    times = total_time[tract_idxs, res_idxs]
    weights = cp.round(1.0 / cp.maximum(times, 0.5), 4)

    src_nodes = tr_nids[tract_idxs].get().tolist()
    dst_nodes = r_nids[res_idxs].get().tolist()
    wts       = weights.get().tolist()

    sbs.extend(src_nodes); sbd.extend(dst_nodes); sbw.extend(wts)

add(sbs, sbd, sbw, "SERVED_BY")
print(f"    {len(sbs)} SERVED_BY edges  ({time.time()-t0_sb:.0f}s)")

print(f"\nTotal edges: {len(edge_src)}")

## Cell 5 — Build cuGraph object + save

In [ ]:
print("Building cuGraph...")
t0 = time.time()

edges_df = pd.DataFrame({
    "src":    edge_src,
    "dst":    edge_dst,
    "weight": edge_weight,
    "edge_type": edge_type,
})

gdf = cudf.DataFrame({
    "src":    edges_df["src"].values,
    "dst":    edges_df["dst"].values,
    "weight": edges_df["weight"].values,
})

G = cugraph.Graph(directed=True)
G.from_cudf_edgelist(gdf, source="src", destination="dst",
                     edge_attr="weight", renumber=False)

print(f"cuGraph nodes: {G.number_of_nodes()}  edges: {G.number_of_edges()}")
print(f"Build time: {time.time()-t0:.0f}s")

# Save to Drive
out = DATA_DIR / "graph.pkl"
payload = {
    "graph":     G,
    "resources": resources,
    "transit":   transit,
    "tracts":    tracts,
    "edges":     edges_df,
    "backend":   "cugraph",
    "offsets": {
        "resource": RESOURCE_OFFSET,
        "transit":  TRANSIT_OFFSET,
        "tract":    TRACT_OFFSET,
    },
}

with open(out, "wb") as f:
    pickle.dump(payload, f)

mb = out.stat().st_size / 1e6
print(f"\n✓ Saved graph.pkl → {out}  ({mb:.1f} MB)")
print("\nEdge type breakdown:")
for etype, cnt in edges_df['edge_type'].value_counts().items():
    print(f"  {etype:<20} {cnt:>8,}")

## Cell 6 — Quick sanity checks

In [ ]:
# Test 1: BFS from a random shelter
print("=== Test 1: BFS from a shelter ===")
shelter_rows = resources[resources['resource_type'] == 'shelter']
sample_node = int(shelter_rows.iloc[0]['node_id'])
shelter_name = shelter_rows.iloc[0].get('name', 'Unknown')
print(f"Starting BFS from: {shelter_name} (node {sample_node})")

bfs_result = cugraph.bfs(G, start=sample_node)
bfs_pd = bfs_result.to_pandas()
reachable = bfs_pd[bfs_pd['distance'] < 1e9]
print(f"Reachable nodes: {len(reachable)} / {G.number_of_nodes()} total")
print(f"Max BFS depth: {reachable['distance'].max()}")

# Test 2: SSSP from a tract to find nearest resources
print("\n=== Test 2: SSSP from a census tract ===")
sample_tract = int(tracts.iloc[0]['node_id'])
tract_name = tracts.iloc[0]['tract_id']
print(f"SSSP from tract: {tract_name} (node {sample_tract})")

sssp = cugraph.sssp(G, source=sample_tract)
sssp_pd = sssp.to_pandas()

# Find the 5 nearest shelters
shelter_node_ids = set(shelter_rows['node_id'].tolist())
shelter_dists = sssp_pd[sssp_pd['vertex'].isin(shelter_node_ids)].nsmallest(5, 'distance')
print("5 nearest shelters by transit time:")
for _, row in shelter_dists.iterrows():
    r = resources[resources['node_id'] == row['vertex']]
    name = r.iloc[0]['name'] if len(r) else 'Unknown'
    print(f"  {name[:40]:<40} dist={row['distance']:.2f}")

print("\n✓ Graph validation passed — ready for hackathon!")